# 📊 Análisis Exploratorio de Datos (EDA)
## Clasificación de Calidad de Frutas
**Universidad Icesi · APO III · 2026-1**

Este notebook corresponde a la **Fase 2 (Comprensión de los Datos)** del ciclo CRISP-DM.

### Objetivos:
1. Verificar la distribución de clases (fruta × calidad) y detectar desbalance.
2. Visualizar ejemplos representativos de cada categoría.
3. Analizar los histogramas de color (HSV) promedio por tipo de fruta.
4. Verificar el pipeline de preprocesamiento cargando un batch de prueba.
5. Documentar hallazgos relevantes para el informe final.

## 0. Configuración e Importaciones

In [ ]:
import sys
import os

# Asegurar que la raíz del proyecto esté en el path de Python
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from PIL import Image

from src.data.preprocess import (
    FruitDatasetBuilder,
    FRUIT_CLASSES,
    QUALITY_CLASSES,
    extract_hsv_histogram,
)

# Configuración de matplotlib
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'text.color':       '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'grid.color':       '#444466',
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

RESULTS_DIR = Path('../experiments/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Importaciones correctas. Listo para el EDA.')

## 1. Construcción del Inventario y Resumen del Dataset

In [ ]:
# Construir el inventario SIN balanceo para ver la distribución real
builder = FruitDatasetBuilder(balance=False)
builder.build()
builder.summary()

inventory = builder.inventory_full
print(f'Total de imágenes en el inventario (sin balanceo): {len(inventory)}')

## 2. Distribución de Clases — Detección de Desbalance

In [ ]:
# Crear DataFrame para análisis
df = pd.DataFrame(inventory)

# ── Figura 1: Distribución por Fruta y Calidad ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Distribución del Dataset de Kaggle', fontsize=16, fontweight='bold', y=1.02)

# Colores por calidad
quality_colors = {'Bueno': '#4ade80', 'Malo': '#f87171', 'Normal': '#60a5fa'}

# Panel izquierdo: count por fruta
ax = axes[0]
fruit_counts = df.groupby(['fruit', 'quality']).size().unstack(fill_value=0)
fruit_counts.plot(
    kind='bar', ax=ax,
    color=[quality_colors.get(q, '#999') for q in fruit_counts.columns],
    edgecolor='#333', linewidth=0.8
)
ax.set_title('Imágenes por Fruta y Calidad', fontsize=13)
ax.set_xlabel('Tipo de Fruta')
ax.set_ylabel('Número de Imágenes')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Calidad', loc='upper right')
ax.grid(axis='y')

# Panel derecho: distribución total por calidad (pie chart)
ax2 = axes[1]
quality_totals = df['quality'].value_counts()
wedge_colors = [quality_colors.get(q, '#999') for q in quality_totals.index]
wedges, texts, autotexts = ax2.pie(
    quality_totals.values,
    labels=quality_totals.index,
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
    wedgeprops=dict(edgecolor='#1a1a2e', linewidth=2)
)
for at in autotexts:
    at.set_color('#1a1a2e')
    at.set_fontweight('bold')
ax2.set_title('Distribución Total por Calidad', fontsize=13)

plt.tight_layout()
plt.savefig(RESULTS_DIR / '01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en experiments/results/01_class_distribution.png')

In [ ]:
# ── Heatmap de conteos (tabla de calor) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
pivot = df.groupby(['fruit', 'quality']).size().unstack(fill_value=0)

sns.heatmap(
    pivot, annot=True, fmt='d',
    cmap='YlOrRd', ax=ax,
    linewidths=0.5, linecolor='#333',
    cbar_kws={'label': 'Número de imágenes'}
)
ax.set_title('Mapa de Calor: Imágenes por Fruta × Calidad', fontsize=14, pad=15)
ax.set_xlabel('Calidad')
ax.set_ylabel('Tipo de Fruta')
plt.tight_layout()
plt.savefig(RESULTS_DIR / '02_heatmap_counts.png', dpi=150, bbox_inches='tight')
plt.show()

# Imprimir la clase más desbalanceada
print('\n⚠️  DESBALANCE DETECTADO:')
print(pivot)

## 3. Ejemplos Visuales por Categoría

In [ ]:
import random
random.seed(42)

# Mostrar 3 ejemplos por fruta para cada calidad disponible
COLS_PER_QUALITY = 3
qualities_present = df['quality'].unique()
n_cols = COLS_PER_QUALITY * len(qualities_present)
n_rows = len(FRUIT_CLASSES)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
fig.suptitle('Ejemplos de Imágenes por Fruta y Calidad', fontsize=16, fontweight='bold')

for row_i, fruit in enumerate(FRUIT_CLASSES):
    for col_qi, quality in enumerate(qualities_present):
        subset = df[(df['fruit'] == fruit) & (df['quality'] == quality)]
        samples = subset.sample(min(COLS_PER_QUALITY, len(subset)), random_state=42)

        for col_j, (_, item) in enumerate(samples.iterrows()):
            ax_col = col_qi * COLS_PER_QUALITY + col_j
            ax = axes[row_i][ax_col] if n_rows > 1 else axes[ax_col]

            img = Image.open(item['path']).convert('RGB').resize((128, 128))
            ax.imshow(img)
            ax.axis('off')

            # Título solo en la primera fila
            if row_i == 0:
                ax.set_title(f'{quality}\n{col_j+1}',
                             color=quality_colors.get(quality, '#fff'),
                             fontsize=9, fontweight='bold')

        # Etiqueta de fruta al inicio de la fila
        if col_qi == 0:
            axes[row_i][0].set_ylabel(fruit, fontsize=11,
                                       fontweight='bold', rotation=90,
                                       labelpad=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / '03_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada en experiments/results/03_sample_images.png')

## 4. Análisis de Histogramas de Color Promedio por Fruta (Canal H de HSV)

El canal **H (Hue / Matiz)** del espacio HSV es el descriptor de color más informativo para distinguir tipos de fruta (ej. las naranjas tienen H≈15°, los bananos H≈30°, las manzanas rojas H≈0° o H≈170°). Analizar el Hue permite también detectar el estado de madurez (frutas madras tienden a desplazarse hacia tonos más cálidos).

In [ ]:
# Calcular el histograma H promedio para una muestra de cada fruta + calidad
SAMPLE_N = 100  # Muestras por combinación (fruta, calidad) para el cálculo
N_BINS = 32
BINS_X = np.linspace(0, 180, N_BINS)

def avg_hue_histogram(paths, n_bins=32):
    """Calcula el histograma de Hue promedio para una lista de rutas de imagen."""
    hists = []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.resize(img, (128, 128))
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        h_hist = cv2.calcHist([hsv], [0], None, [n_bins], [0, 180]).flatten()
        h_hist = h_hist / (h_hist.sum() + 1e-7)
        hists.append(h_hist)
    return np.mean(hists, axis=0) if hists else np.zeros(n_bins)

fig, axes = plt.subplots(len(FRUIT_CLASSES), 1,
                          figsize=(14, len(FRUIT_CLASSES) * 2.5), sharex=True)
fig.suptitle('Histograma de Hue (H) Promedio por Fruta y Calidad',
             fontsize=15, fontweight='bold')

for ax, fruit in zip(axes, FRUIT_CLASSES):
    for quality in qualities_present:
        subset_paths = df[(df['fruit'] == fruit) & (df['quality'] == quality)]['path'].tolist()
        sample_paths = random.sample(subset_paths, min(SAMPLE_N, len(subset_paths)))
        hist = avg_hue_histogram(sample_paths, n_bins=N_BINS)
        color = quality_colors.get(quality, '#999')
        ax.fill_between(BINS_X, hist, alpha=0.55, color=color, label=quality)
        ax.plot(BINS_X, hist, color=color, linewidth=1.5)

    ax.set_ylabel(fruit, fontsize=10, fontweight='bold', rotation=0, labelpad=60)
    ax.grid(True)
    ax.set_ylim(bottom=0)
    if fruit == FRUIT_CLASSES[0]:
        ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Valor del canal Hue (0° – 180°)', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / '04_hue_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en experiments/results/04_hue_histograms.png')

## 5. Verificación del Pipeline de Preprocesamiento

In [ ]:
# Reconstruir con balanceo para verificar el pipeline completo
builder_balanced = FruitDatasetBuilder(balance=True)
builder_balanced.build()
builder_balanced.summary()

# Obtener DataLoaders de PyTorch
train_loader, val_loader, test_loader = builder_balanced.get_dataloaders(batch_size=16)

# Tomar un batch de muestra
imgs, fruit_labels, quality_labels = next(iter(train_loader))

print(f'Shape del batch    : {imgs.shape}  → (batch, canales, alto, ancho)')
print(f'Rango de valores   : [{imgs.min():.3f}, {imgs.max():.3f}]  (normalizado)')
print(f'Frutas en el batch : {[FRUIT_CLASSES[i] for i in fruit_labels]}')
print(f'Calidades          : {[QUALITY_CLASSES[i] for i in quality_labels]}')

In [ ]:
# Visualizar el batch augmentado (des-normalizar para mostrar)
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    img = tensor.numpy().transpose(1, 2, 0)
    img = img * std + mean
    return np.clip(img, 0, 1)

n_show = min(8, len(imgs))
fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.2, 2.8))
fig.suptitle('Batch de Entrenamiento (con Data Augmentation aplicado)', fontsize=13)

for i in range(n_show):
    ax = axes[i]
    ax.imshow(denormalize(imgs[i]))
    fruit_name   = FRUIT_CLASSES[fruit_labels[i]]
    quality_name = QUALITY_CLASSES[quality_labels[i]]
    ax.set_title(f'{fruit_name}\n{quality_name}',
                 fontsize=8,
                 color=quality_colors.get(quality_name, '#fff'))
    ax.axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / '05_augmented_batch.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en experiments/results/05_augmented_batch.png')

## 6. Hallazgos y Conclusiones del EDA

*(Esta sección se debe completar en el informe final con base en los resultados observados en las figuras)*

### Distribución de Clases
- El dataset de Kaggle contiene solo las categorías **Bueno** y **Malo**.
- **Granada (Pomegranate) Bueno** presenta un desbalance significativo (~5940 imágenes) frente a las demás clases (~1085–1216 imágenes). Sin corrección, el clasificador aprendería a sesgar sus predicciones hacia esta clase.
- **Estrategia aplicada**: Submuestreo aleatorio al tamaño de la clase minoritaria para garantizar un entrenamiento justo.

### Análisis de Color (Canal H de HSV)
- Las frutas exhiben distribuciones de Hue claramente diferenciadas:
  - 🍊 **Naranja** y 🍌 **Banano**: Hue concentrado en tonos cálidos (10°–40°).
  - 🍈 **Guayaba**: Distribución amplia con mezcla de verdes y amarillos.
  - 🍎 **Manzana** y 🍎 **Granada**: Alta variabilidad (verdes, rojas, amarillas).
- Las frutas **Malas** tienden a presentar desplazamientos en el Hue hacia tonos más oscuros/pardos, lo que justifica el uso de histogramas de color como características para los modelos ML.

### Calidad de Imágenes
- Todas las imágenes tienen fondo uniforme (condición controlada), lo que facilita la segmentación.
- La resolución es variable; el pipeline normaliza a **128×128 px** para la CNN.